## Graph WaveNet — Model Architecture
`model.ipynb` | Tomaz Cavalcante

Implementation of **Graph WaveNet for Deep Spatial-Temporal Graph Modeling** (Wu et al., IJCAI 2019).

**Three building blocks, assembled in order:**
1. `CausalConv2d` — dilated 1D conv that only looks at past timesteps
2. `WaveNetBlock` — gated temporal conv + graph conv + skip/residual connections
3. `GraphWaveNet` — stacks all blocks; learns an adaptive graph on top of the static sensor graph

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {DEVICE}')

device: cuda


### 1 · Causal Dilated Convolution

Standard `Conv2d` pads symmetrically, so the model would see future timesteps — a data leak.
`CausalConv2d` pads **only the left (past) side** so each output at time `t` depends only on `t' <= t`.

The `dilation` parameter exponentially expands the receptive field without adding parameters:

| dilation | receptive field (kernel=2) |
|----------|---------------------------|
| 1 | 2 steps |
| 2 | 4 steps |
| 4 | 8 steps |
| 8 | 16 steps |

With `n_blocks=4, n_layers=2` the dilation cycle `[1, 2, 1, 2, 1, 2, 1, 2]`
gives a total receptive field of **13 timesteps** — just enough to cover `seq_len=12`.

In [ ]:
class CausalConv2d(nn.Module):
    # Dilated causal 1-D convolution over the Time dimension.
    # Input / output shape: [Batch, Channels, Nodes, Time]  (T unchanged)

    def __init__(self, in_channels, out_channels, kernel_size=2, dilation=1):
        super().__init__()
        # Number of positions to pad on the LEFT to maintain causality
        self.causal_pad = (kernel_size - 1) * dilation
        # kernel=(1, K): mixes only across time, NOT across nodes
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=(1, kernel_size),
            dilation=(1, dilation),
            padding=0,
        )

    def forward(self, x):
        # x: [B, C, N, T]
        if self.causal_pad > 0:
            x = F.pad(x, (self.causal_pad, 0))  # pad T dim: (left=past, right=0)
        return self.conv(x)  # T dimension unchanged

### 2 · WaveNet Block

Each block performs two operations in sequence, then adds residual and skip connections.

**a) Gated temporal convolution** — WaveNet's signature:
```
h = CausalConv(x)                           # [B, 2C, N, T]
out = tanh(h[:C]) * sigmoid(h[C:])          # gated activation
```
The sigmoid gate learns which timesteps are relevant (values near 0 are suppressed).

**b) Graph convolution — K-hop aggregation**

For each adjacency matrix A, collect multi-hop features and project:
```
h = Linear( concat([x, A·x, A²·x]) )        # K=2 hops
```
Three adjacency matrices are used: **forward** `A_fw`, **backward** `A_bw`, **adaptive** `A_adp`.

The residual connection preserves gradients through many stacked blocks (like ResNet).
The skip connection routes a portion of each block's signal directly to the output head.

In [ ]:
class WaveNetBlock(nn.Module):
    # One stacked layer: gated causal conv (temporal) + graph conv (spatial).
    # In / out shape : [B, residual_channels, N, T]
    # Skip output    : [B, skip_channels,     N, T]

    def __init__(
        self,
        residual_channels,   # channel width inside the block
        skip_channels,       # width of skip-connection output
        kernel_size,         # temporal conv kernel size (typically 2)
        dilation,            # temporal dilation factor (1, 2, 4, ...)
        num_adj_matrices,    # number of adjacency matrices (3: fw, bw, adp)
        gcn_depth,           # K for K-hop aggregation (typically 2)
        dropout=0.3,
    ):
        super().__init__()
        self.gcn_depth = gcn_depth

        # Gated causal conv: outputs 2x channels, then split into filter + gate
        self.temporal_conv = CausalConv2d(
            residual_channels, 2 * residual_channels, kernel_size, dilation
        )

        # Graph conv: for each adj matrix collect (K+1) hops of residual_channels
        gcn_in = residual_channels * (gcn_depth + 1) * num_adj_matrices
        self.spatial_proj = nn.Linear(gcn_in, residual_channels)

        # 1x1 convolutions for skip and residual paths
        self.skip_conv     = nn.Conv2d(residual_channels, skip_channels,     kernel_size=1)
        self.residual_conv = nn.Conv2d(residual_channels, residual_channels, kernel_size=1)
        self.norm          = nn.BatchNorm2d(residual_channels)
        self.dropout       = nn.Dropout(dropout)

    def forward(self, x, adj_list):
        # x: [B, C, N, T]
        residual = x  # saved for the residual connection at the end

        # -- Step 1: Gated temporal convolution --------------------------------
        h = self.temporal_conv(x)                   # [B, 2C, N, T]
        h_filter, h_gate = h.chunk(2, dim=1)        # each: [B, C, N, T]
        h_tcn = torch.tanh(h_filter) * torch.sigmoid(h_gate)

        # -- Step 2: Graph convolution (K-hop neighbourhood aggregation) -------
        B, C, N, T = h_tcn.shape
        hop_features = []
        for A in adj_list:                          # A: [N, N]
            h_k = h_tcn
            for k in range(self.gcn_depth + 1):     # 0-hop, 1-hop, 2-hop ...
                hop_features.append(h_k)
                if k < self.gcn_depth:
                    # A: [N, N] | h_k: [B, C, N, T]  -> aggregate over N
                    h_k = torch.einsum('nm, bcmt -> bcnt', A, h_k)

        h_cat = torch.cat(hop_features, dim=1)      # [B, C*(K+1)*#adj, N, T]
        h_cat = h_cat.permute(0, 3, 2, 1)           # [B, T, N, C*(K+1)*#adj]
        h_gcn = F.relu(self.spatial_proj(h_cat))         # [B, T, N, residual_channels]
        h_gcn = h_gcn.permute(0, 3, 2, 1)                   # [B, residual_channels, N, T]
        h_gcn = self.dropout(h_gcn)

        h = h_gcn + h_tcn

        # -- Step 3: Skip and residual connections -----------------------------
        skip = self.skip_conv(h)                     # [B, skip_channels, N, T]
        h    = self.norm(self.residual_conv(h) + residual)

        return h, skip

### 3 · Graph WaveNet (Full Model)

```
x  [B, 1, N, 12]
    |
 input_proj  Conv2d(1 -> 32)
    |
    +-- WaveNetBlock  dilation=1 ----> skip
    +-- WaveNetBlock  dilation=2 ----> skip    (x n_blocks cycles)
    +-- WaveNetBlock  dilation=1 ----> skip
    +-- WaveNetBlock  dilation=2 ----> skip
    ...
         sum all skip[..., -1:]
              |
         [B, 256, N, 1]
              |
         ReLU -> Conv(256->512) -> ReLU -> Conv(512->12)
              |
         [B, N, 12]     <- one speed prediction per node per horizon step
```

**Adaptive adjacency matrix** — the key innovation of Graph WaveNet:
```python
A_adp = softmax( ReLU( E1 @ E2^T ) )   # shape: [N, N]
```
`E1`, `E2` are `[N, embed_dim]` **learnable** parameters. Their outer product scores the
affinity between every pair of nodes — the model discovers hidden correlations not present
in the physical sensor graph.

In [ ]:
class GraphWaveNet(nn.Module):
    # Graph WaveNet: Wu et al., IJCAI 2019
    # https://arxiv.org/abs/1906.00121
    #
    # Args:
    #   num_nodes         : graph nodes (207 METR-LA, 325 PEMS-BAY)
    #   in_channels       : input features per node per step (1 = speed only)
    #   out_channels      : forecast horizon steps (12 = 1 hour at 5 min)
    #   residual_channels : hidden width inside each block
    #   skip_channels     : accumulated skip signal width
    #   end_channels      : intermediate FC width in output head
    #   kernel_size       : temporal conv kernel (paper default: 2)
    #   n_blocks          : number of dilation cycles
    #   n_layers          : layers per cycle; dilations = 2^0, 2^1, ..., 2^(n_layers-1)
    #   gcn_depth         : K in K-hop graph aggregation
    #   dropout           : dropout probability inside blocks
    #   embed_dim         : node embedding size for adaptive adjacency

    def __init__(
        self,
        num_nodes,
        in_channels       = 1,
        out_channels      = 12,
        residual_channels = 32,
        skip_channels     = 256,
        end_channels      = 512,
        kernel_size       = 2,
        n_blocks          = 4,
        n_layers          = 2,
        gcn_depth         = 2,
        dropout           = 0.3,
        embed_dim         = 10,
    ):
        super().__init__()
        self.skip_channels = skip_channels
        num_adj = 3  # forward + backward + adaptive

        # -- Learnable node embeddings for adaptive adjacency ------------------
        # A_adp = softmax(ReLU(E1 @ E2^T))  shape: [N, N]
        self.node_emb1 = nn.Parameter(torch.randn(num_nodes, embed_dim))
        self.node_emb2 = nn.Parameter(torch.randn(num_nodes, embed_dim))

        # -- Input projection: raw features -> residual_channels ---------------
        self.input_proj = nn.Conv2d(in_channels, residual_channels, kernel_size=1)

        # -- Stacked WaveNet + GCN blocks --------------------------------------
        # Dilation schedule example (n_blocks=4, n_layers=2):
        #   cycle 1: dilation 1, 2
        #   cycle 2: dilation 1, 2
        #   cycle 3: dilation 1, 2
        #   cycle 4: dilation 1, 2   -> total 8 layers
        self.blocks = nn.ModuleList()
        for _ in range(n_blocks):
            for i in range(n_layers):
                self.blocks.append(WaveNetBlock(
                    residual_channels = residual_channels,
                    skip_channels     = skip_channels,
                    kernel_size       = kernel_size,
                    dilation          = 2 ** i,
                    num_adj_matrices  = num_adj,
                    gcn_depth         = gcn_depth,
                    dropout           = dropout,
                ))

        # -- Output head: skip sum -> multi-step forecast ----------------------
        self.out_proj1 = nn.Conv2d(skip_channels, end_channels,  kernel_size=1)
        self.out_proj2 = nn.Conv2d(end_channels,  out_channels,  kernel_size=1)

    def _adaptive_adj(self, device):
        # Row-wise softmax ensures values in (0, 1) and each row sums to 1
        return torch.softmax(
            F.relu(self.node_emb1 @ self.node_emb2.T), dim=1
        ).to(device)

    def forward(self, x, static_adj):
        # x          : [B, F, N, T]
        # static_adj : [N, N] sparse tensor (from load_adj_matrix in utilities.ipynb)
        # returns    : [B, N, out_channels]

        device = x.device
        A_fw   = static_adj.to_dense().to(device)   # static forward
        A_bw   = A_fw.T.contiguous()                # static backward (transpose)
        A_adp  = self._adaptive_adj(device)         # learned
        adj_list = [A_fw, A_bw, A_adp]

        h = self.input_proj(x)   # [B, residual_channels, N, T]

        # Accumulate skip connections across all blocks.
        # skip_sum starts as 0 (broadcast to correct shape on first add).
        skip_sum = 0
        for block in self.blocks:
            h, skip = block(h, adj_list)
            # Take only the LAST timestep from each skip (it has seen all context)
            skip_sum = skip_sum + skip[..., -1:]    # [B, skip_channels, N, 1]

        out = F.relu(skip_sum)                      # [B, skip_channels, N, 1]
        out = F.relu(self.out_proj1(out))            # [B, end_channels,  N, 1]
        out = self.out_proj2(out)                    # [B, out_channels,  N, 1]
        out = out.squeeze(-1).permute(0, 2, 1)      # [B, N, out_channels]

        return out

### Sanity Check

Verifies:
- The model initializes and all layer dimensions are consistent
- A forward pass produces the expected output shape `[B, N, 12]`
- Parameter count is reasonable (~300K for default settings)

A **random normalized adjacency matrix** is used here — no data files needed.

In [ ]:
NUM_NODES    = 207   # METR-LA  (use 325 for PEMS-BAY)
IN_CHANNELS  = 1    # speed only
OUT_CHANNELS = 12   # 12-step horizon = 1 hour at 5-min intervals

model = GraphWaveNet(
    num_nodes         = NUM_NODES,
    in_channels       = IN_CHANNELS,
    out_channels      = OUT_CHANNELS,
    residual_channels = 32,
    skip_channels     = 256,
    end_channels      = 512,
    kernel_size       = 2,
    n_blocks          = 4,
    n_layers          = 2,
    gcn_depth         = 2,
    dropout           = 0.3,
    embed_dim         = 10,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

# Build a dummy row-normalised adjacency (no data files needed)
torch.manual_seed(0)
dense_adj = torch.rand(NUM_NODES, NUM_NODES)
dense_adj = (dense_adj + dense_adj.T) / 2                  # make symmetric
row_sums  = dense_adj.sum(dim=1, keepdim=True).clamp(1e-6)
norm_adj  = (dense_adj / row_sums).to_sparse().to(DEVICE)

# Forward pass
x_test = torch.randn(4, IN_CHANNELS, NUM_NODES, 12).to(DEVICE)

with torch.no_grad():
    out = model(x_test, norm_adj)

print(f'Input  shape : {x_test.shape}')   # [4, 1, 207, 12]
print(f'Output shape : {out.shape}')      # [4, 207, 12]
assert out.shape == (4, NUM_NODES, OUT_CHANNELS), 'Shape mismatch!'
print('All shape checks passed.')

Trainable parameters: 325,752
Input  shape : torch.Size([4, 1, 207, 12])
Output shape : torch.Size([4, 207, 12])
All shape checks passed.
